## Experiment: Does QLoRA Fine-Tuning Help an LLM Write in Your Personal Chat Style?

This notebook compares three variants of the same small instruction-tuned model on identical held-out conversation turns:

1. **Base** — short style instruction only,
2. **Few-shot** — a few real examples sampled exclusively from the `train` split,
3. **QLoRA** — a 4-bit LoRA adapter fine-tuned only on your target replies (`-100` loss masking on all prompt/context tokens).

**Privacy:** All data preprocessing, PII scrubbing, training, and inference run locally. Only anonymized texts enter the training pipeline, and nothing is sent to external APIs. Only public base model weights are downloaded from Hugging Face.


In [ ]:
# ===== Experiment Configuration =====
from __future__ import annotations
import datetime as dt
import gc
import html
import json
import math
import random
import re
import time
import warnings
import zipfile
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")

# Replace with the exact senderName strings from your JSON export.
ME = "Speaker A"
FRIEND = "Speaker B"

# Path to a Messenger-style .json export or .zip archive containing one.
# Expected JSON schema: {"messages": [{"senderName": str, "timestamp": int_ms, "text": str, "isUnsent": bool}]}
DATA_PATH = Path("data/messages.json")

# Max pause (seconds) within which consecutive messages by the same author merge into one utterance.
MERGE_GAP_SECONDS = 3 * 60

# Context window: up to 8 prior utterances, at least 3, and at least one by ME.
MAX_CONTEXT_UTTERANCES = 8
MIN_CONTEXT_UTTERANCES = 3

# Strict chronological split: the most recent 10% of history is reserved for test.
SPLIT_FRACTIONS = {"train": 0.80, "validation": 0.10, "test": 0.10}
SPLIT_ORDER = ["train", "validation", "test"]

# Small, ungated instruction model; 4-bit QLoRA fits comfortably on an 8 GB VRAM GPU.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

FEWSHOT_N = 3
N_TEST_CASES = 6
MAX_SEQ_LEN = 640
MAX_CONTEXT_TOKENS = 1280
SEED = 42

# Shared generation parameters for all three variants.
GEN_KWARGS = {
    "max_new_tokens": 96,
    "do_sample": False,
    "repetition_penalty": 1.1,
    "no_repeat_ngram_size": 4,
}

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
pd.set_option("display.max_colwidth", 160)

print("Configuration ready.")
print("Model:", MODEL_ID)
print("Utterance merge gap:", MERGE_GAP_SECONDS, "s")
print("Chronological split:", SPLIT_FRACTIONS)


In [ ]:
# ===== Load Export (JSON / ZIP) =====
def load_chat_data(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Chat export not found at '{path}'. Place your Messenger-style JSON/ZIP export there or update DATA_PATH."
        )
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            json_candidates = [
                (info.file_size, info.filename)
                for info in zf.infolist()
                if info.filename.lower().endswith(".json")
            ]
            if not json_candidates:
                raise FileNotFoundError(f"No .json files found inside {path}")
            _, target_name = max(json_candidates)
            with zf.open(target_name) as f:
                return json.load(f)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


data = load_chat_data(DATA_PATH)
print("Loaded raw messages:", len(data["messages"]))


In [ ]:
# ===== PII Anonymization & Message Filtering =====
PLACEHOLDER_ME = "[ME]"
PLACEHOLDER_FRIEND = "[FRIEND]"
PLACEHOLDER_NAME = "[NAME]"
PLACEHOLDER_FIRST_NAME = "[FIRST_NAME]"
PLACEHOLDER_EMAIL = "[EMAIL]"
PLACEHOLDER_PHONE = "[PHONE]"
PLACEHOLDER_URL = "[URL]"
PLACEHOLDER_HANDLE = "[HANDLE]"

_EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b")
_URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)\S+")
_BARE_DOMAIN_RE = re.compile(
    r"(?i)(?<![\w@])(?:[a-z0-9\-]+\.)+(?:pl|com|net|org|io|eu|de|uk|se|app|dev)(?:/\S*)?"
)
_PHONE_RE = re.compile(r"(?<![\w])(?:\+?\d[\d\s().\-]{6,}\d)(?![\w])")
_HANDLE_RE = re.compile(r"(?<![\w])@[A-Za-z0-9_]{2,}")

# Extend this set with common first names for your language/region so "First Last" pairs
# and standalone first names are scrubbed without stripping ordinary capitalized words.
_COMMON_FIRST_NAMES = {
    "Adam", "Alex", "Alice", "Amanda", "Andrew", "Anna", "Anthony", "Benjamin",
    "Caroline", "Catherine", "Charles", "Charlotte", "Chris", "Christian", "Claire",
    "Daniel", "David", "Dominic", "Elizabeth", "Emily", "Emma", "Eric", "Ethan",
    "Eva", "Gabriel", "George", "Grace", "Hannah", "Henry", "Isabel", "Jack",
    "Jacob", "James", "Jason", "Jessica", "John", "Jonathan", "Joseph", "Julia",
    "Justin", "Kate", "Katherine", "Kevin", "Laura", "Lauren", "Liam", "Lucas",
    "Luke", "Mark", "Martin", "Mary", "Matthew", "Michael", "Natalie", "Nathan",
    "Nicholas", "Noah", "Oliver", "Olivia", "Oscar", "Patrick", "Paul", "Peter",
    "Philip", "Rachel", "Rebecca", "Richard", "Robert", "Ryan", "Samuel", "Sarah",
    "Sebastian", "Simon", "Sophia", "Sophie", "Stephen", "Thomas", "Tom", "Victor",
    "Victoria", "William",
}
_FIRST_NAME_ALT = "|".join(
    re.escape(name) for name in sorted(_COMMON_FIRST_NAMES, key=len, reverse=True)
)
_NAME_PAIR_RE = re.compile(
    rf"\b(?:{_FIRST_NAME_ALT})\s+[A-Z][a-z]{{2,}}\b"
)
_FIRST_NAME_RE = re.compile(rf"\b(?:{_FIRST_NAME_ALT})\b")
_LOWERCASE_FIRST_NAME_RE = re.compile(
    rf"(?i:\b(?:{_FIRST_NAME_ALT})\b)"
)
_LABELED_SURNAME_RE = re.compile(
    r"(?<!\[)((?i:Surname|Last\s+name)\s*[?:]?\s*)([A-Z][a-z]{2,})\b"
)

def _build_participant_patterns(me: str, friend: str) -> list[tuple[re.Pattern, str]]:
    patterns = []
    for full_name, placeholder in [(me, PLACEHOLDER_ME), (friend, PLACEHOLDER_FRIEND)]:
        tokens = [re.escape(tok) for tok in full_name.strip().split() if len(tok) >= 2]
        if len(tokens) >= 2:
            patterns.append((re.compile(r"\b" + r"\s+".join(tokens) + r"\b", re.I), placeholder))
        for tok in sorted(tokens, key=len, reverse=True):
            patterns.append((re.compile(rf"\b{tok}\b", re.I), placeholder))
    return patterns


_NAME_REPLACEMENTS = _build_participant_patterns(ME, FRIEND)


def _replace_phone(match: re.Match) -> str:
    value = match.group(0)
    return PLACEHOLDER_PHONE if sum(ch.isdigit() for ch in value) >= 9 else value


def anonymize_text(text) -> str:
    """Scrubs detectable PII while preserving informal tone, slang, punctuation, and line breaks."""
    if text is None:
        return ""
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = _URL_RE.sub(PLACEHOLDER_URL, text)
    text = _BARE_DOMAIN_RE.sub(PLACEHOLDER_URL, text)
    text = _EMAIL_RE.sub(PLACEHOLDER_EMAIL, text)
    text = _PHONE_RE.sub(_replace_phone, text)
    text = _HANDLE_RE.sub(PLACEHOLDER_HANDLE, text)
    for pattern, replacement in _NAME_REPLACEMENTS:
        text = pattern.sub(replacement, text)
    text = _NAME_PAIR_RE.sub(PLACEHOLDER_NAME, text)
    text = _LABELED_SURNAME_RE.sub(lambda m: m.group(1) + PLACEHOLDER_NAME, text)
    text = _FIRST_NAME_RE.sub(PLACEHOLDER_FIRST_NAME, text)
    text = _LOWERCASE_FIRST_NAME_RE.sub(PLACEHOLDER_FIRST_NAME, text)
    return text


def prepare_messages(data: dict) -> tuple[list[dict], Counter]:
    if "messages" not in data:
        raise KeyError("data must contain a 'messages' key")
    rows = []
    skipped = Counter()
    for source_idx, message in enumerate(data["messages"]):
        if bool(message.get("isUnsent")):
            skipped["isUnsent"] += 1
            continue
        text = message.get("text") or ""
        if not str(text).strip():
            skipped["empty_text"] += 1
            continue
        timestamp = message.get("timestamp")
        if timestamp is None:
            skipped["missing_timestamp"] += 1
            continue
        sender = message.get("senderName")
        if sender not in (ME, FRIEND):
            skipped["other_sender"] += 1
            continue
        clean = anonymize_text(str(text))
        if not clean.strip():
            skipped["empty_after_anonymization"] += 1
            continue
        rows.append({
            "source_idx": source_idx,
            "sender": sender,
            "text": clean.strip(),
            "timestamp": int(timestamp),
            "type": message.get("type"),
        })
    rows.sort(key=lambda row: (row["timestamp"], row["source_idx"]))
    return rows, skipped


filtered_messages, skipped_counts = prepare_messages(data)
print("Total messages in export:", len(data["messages"]))
print("Filtered messages:", len(filtered_messages))
print("Skipped:", dict(skipped_counts))


In [ ]:
# ===== Chronological Split & Multi-Message Utterance Merging =====
def fmt_ts(timestamp_ms: int) -> str:
    return dt.datetime.fromtimestamp(timestamp_ms / 1000).strftime("%Y-%m-%d %H:%M")


def chronological_split(messages: list[dict], fractions: dict[str, float]) -> dict[str, list[dict]]:
    n = len(messages)
    n_train = max(1, min(int(n * fractions["train"]), n - 2))
    n_val = max(1, min(int(n * fractions["validation"]), n - n_train - 1))
    parts = {
        "train": messages[:n_train],
        "validation": messages[n_train:n_train + n_val],
        "test": messages[n_train + n_val:],
    }
    for split_name, split_messages in parts.items():
        for message in split_messages:
            message["split"] = split_name
    return parts


def merge_utterances(messages: list[dict], split_name: str,
                     gap_seconds: int = MERGE_GAP_SECONDS) -> list[dict]:
    """Merges consecutive messages from the same sender within gap_seconds into one multi-line turn."""
    utterances = []
    for message in messages:
        same_author = bool(utterances) and utterances[-1]["sender"] == message["sender"]
        short_gap = same_author and (message["timestamp"] - utterances[-1]["end_ts"]) <= gap_seconds * 1000
        if short_gap:
            utterances[-1]["text"] += "\n" + message["text"]
            utterances[-1]["end_ts"] = message["timestamp"]
            utterances[-1]["source_ids"].append(message["source_idx"])
            utterances[-1]["n_messages"] += 1
        else:
            utterances.append({
                "utterance_id": f"{split_name}-{len(utterances):05d}",
                "split": split_name,
                "sender": message["sender"],
                "text": message["text"],
                "start_ts": message["timestamp"],
                "end_ts": message["timestamp"],
                "source_ids": [message["source_idx"]],
                "n_messages": 1,
            })
    # Second pass after merging catches names split across consecutive message bubbles.
    for utterance in utterances:
        utterance["text"] = anonymize_text(utterance["text"])
    return utterances


messages_by_split = chronological_split(filtered_messages, SPLIT_FRACTIONS)
utterances_by_split = {
    split_name: merge_utterances(messages_by_split[split_name], split_name)
    for split_name in SPLIT_ORDER
}

quick_stats = pd.DataFrame([
    {
        "split": split_name,
        "messages": len(messages_by_split[split_name]),
        "utterances": len(utterances_by_split[split_name]),
        "avg_msgs_per_utterance": round(
            len(messages_by_split[split_name]) / max(1, len(utterances_by_split[split_name])), 2
        ),
        "from": fmt_ts(messages_by_split[split_name][0]["timestamp"]),
        "to": fmt_ts(messages_by_split[split_name][-1]["timestamp"]),
    }
    for split_name in SPLIT_ORDER
])
display(quick_stats)

merged_extra = sum(u["n_messages"] - 1 for utts in utterances_by_split.values() for u in utts)
print("Additional messages merged into multi-line utterances:", merged_extra)


In [ ]:
# ===== Context Window & Training Example Construction =====
def build_examples(utterances: list[dict], split_name: str) -> list[dict]:
    """Creates an example only when ME replies immediately after FRIEND."""
    examples = []
    for i, target in enumerate(utterances):
        if target["sender"] != ME:
            continue
        if i == 0 or utterances[i - 1]["sender"] != FRIEND:
            continue
        start = max(0, i - MAX_CONTEXT_UTTERANCES)
        context = utterances[start:i]
        if len(context) < MIN_CONTEXT_UTTERANCES:
            continue
        if not any(u["sender"] == ME for u in context):
            continue
        if any(u["end_ts"] >= target["start_ts"] for u in context):
            continue
        examples.append({
            "example_id": f"{split_name}-{len(examples):05d}",
            "split": split_name,
            "context": context,
            "target": target,
            "date": fmt_ts(target["start_ts"]),
        })
    return examples


examples_by_split = {
    split_name: build_examples(utterances_by_split[split_name], split_name)
    for split_name in SPLIT_ORDER
}
all_examples = [ex for split_name in SPLIT_ORDER for ex in examples_by_split[split_name]]

split_stats = pd.DataFrame([
    {
        "split": split_name,
        "messages": len(messages_by_split[split_name]),
        "utterances": len(utterances_by_split[split_name]),
        "examples": len(examples_by_split[split_name]),
        "avg_context_utterances": round(
            sum(len(ex["context"]) for ex in examples_by_split[split_name])
            / max(1, len(examples_by_split[split_name])), 1
        ),
        "from": fmt_ts(messages_by_split[split_name][0]["timestamp"]),
        "to": fmt_ts(messages_by_split[split_name][-1]["timestamp"]),
    }
    for split_name in SPLIT_ORDER
])
display(split_stats)

print("Total examples:", len(all_examples))
print(
    "Target reply length (chars): median",
    int(pd.Series([len(ex["target"]["text"]) for ex in all_examples]).median()),
    "| max",
    max(len(ex["target"]["text"]) for ex in all_examples),
)


In [ ]:
# ===== Integrity Assertions: Authorship, Chronology, No Split Leakage, PII Check =====
def _source_ids(utterances: list[dict]) -> set[int]:
    if not utterances:
        return set()
    return set().union(*(set(u["source_ids"]) for u in utterances))


split_source_ids = {
    split_name: _source_ids(utterances_by_split[split_name])
    for split_name in SPLIT_ORDER
}
assert split_source_ids["train"].isdisjoint(split_source_ids["validation"])
assert split_source_ids["train"].isdisjoint(split_source_ids["test"])
assert split_source_ids["validation"].isdisjoint(split_source_ids["test"])

train_ts = [u["start_ts"] for u in utterances_by_split["train"]]
val_ts = [u["start_ts"] for u in utterances_by_split["validation"]]
test_ts = [u["start_ts"] for u in utterances_by_split["test"]]
assert max(train_ts) <= min(val_ts)
assert max(val_ts) <= min(test_ts)

for split_name, examples in examples_by_split.items():
    for ex in examples:
        assert ex["split"] == split_name
        assert ex["target"]["sender"] == ME
        assert ex["context"][-1]["sender"] == FRIEND
        assert MIN_CONTEXT_UTTERANCES <= len(ex["context"]) <= MAX_CONTEXT_UTTERANCES
        assert all(u["split"] == split_name for u in ex["context"])
        assert all(u["end_ts"] < ex["target"]["start_ts"] for u in ex["context"])
        assert set(ex["target"]["source_ids"]).isdisjoint(_source_ids(ex["context"]))

for ex in all_examples:
    for utterance in ex["context"] + [ex["target"]]:
        text = utterance["text"]
        assert ME not in text and FRIEND not in text
        assert not _EMAIL_RE.search(text)
        assert not _URL_RE.search(text)
        assert not _NAME_PAIR_RE.search(text)
        assert not _LABELED_SURNAME_RE.search(text)
        assert not _FIRST_NAME_RE.search(text)
        assert not _LOWERCASE_FIRST_NAME_RE.search(text)

print("Assertions passed: authorship, chronological order, no split leakage, and PII scrubbing verified.")


def format_context(context: list[dict]) -> str:
    lines = []
    for utterance in context:
        speaker = "ME" if utterance["sender"] == ME else "FRIEND"
        lines.append(f"{speaker}: {utterance['text']}")
    return "\n".join(lines)


def _pre(text: str) -> str:
    return html.escape(text).replace("\n", "<br>")


sample_rows = []
for split_name in SPLIT_ORDER:
    if not examples_by_split[split_name]:
        continue
    ex = examples_by_split[split_name][0]
    sample_rows.append(
        "<tr>"
        f"<td><b>{split_name}</b><br><small>{html.escape(ex['date'])}</small></td>"
        f"<td>{_pre(format_context(ex['context']))}</td>"
        f"<td>{_pre(ex['target']['text'])}</td>"
        "</tr>"
    )

display(HTML(
    "<h4>Anonymized Samples (context always ends with FRIEND)</h4>"
    "<table style='border-collapse:collapse;width:100%'>"
    "<tr><th style='text-align:left;padding:6px'>split / date</th>"
    "<th style='text-align:left;padding:6px'>context</th>"
    "<th style='text-align:left;padding:6px'>target reply</th></tr>"
    + "".join(sample_rows) +
    "</table>"
    "<style>td,th{border:1px solid #ccc;vertical-align:top;padding:6px;white-space:pre-wrap}</style>"
))


## Model & Hardware


In [ ]:
# ===== Model, Tokenizer & 4-bit QLoRA Loader =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("CUDA:", torch.cuda.get_device_name(0))
    print("VRAM:", round(props.total_memory / 1e9, 1), "GB | bf16:", torch.cuda.is_bf16_supported())
else:
    print("CUDA unavailable — training will be very slow or infeasible.")


def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok


def make_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )


def load_base_model():
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=make_bnb_config(),
            device_map={"": 0},
            dtype=torch.bfloat16,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
    model.config.use_cache = False
    return model


def load_model_with_optional_adapter(adapter_dir=None):
    model = load_base_model()
    if adapter_dir is not None:
        adapter_dir = Path(adapter_dir)
        if not (adapter_dir / "adapter_config.json").exists():
            raise FileNotFoundError(f"Missing adapter in {adapter_dir}")
        model = PeftModel.from_pretrained(model, str(adapter_dir))
    model.eval()
    return model


tokenizer = load_tokenizer()
print("Tokenizer ready; chat template:", bool(tokenizer.chat_template))
print("EOS:", tokenizer.eos_token, "| PAD:", tokenizer.pad_token)


In [ ]:
# ===== Prompt Formatting, Training Encoding & Target-Only Loss Masking =====
# Adapt the language instruction below if your chat history is in another language.
SYSTEM_PROMPT = (
    "You are a person referred to as ME writing private chat messages to FRIEND. "
    "Mimic the author's natural messaging style: message length, slang, emojis, typos, "
    "punctuation, and line breaks. Reply only with the message content — no commentary, "
    "no metadata, and no invented personal details."
)


def build_chat_messages(context: list[dict], fewshot_examples: list[dict] | None = None) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if fewshot_examples:
        for example in fewshot_examples:
            for utterance in example["context"]:
                role = "user" if utterance["sender"] == FRIEND else "assistant"
                messages.append({"role": role, "content": utterance["text"]})
            messages.append({"role": "assistant", "content": example["target"]["text"]})
    for utterance in context:
        role = "user" if utterance["sender"] == FRIEND else "assistant"
        messages.append({"role": role, "content": utterance["text"]})
    return messages


def prompt_text_for(context: list[dict], fewshot_examples: list[dict] | None = None) -> str:
    return tokenizer.apply_chat_template(
        build_chat_messages(context, fewshot_examples),
        tokenize=False,
        add_generation_prompt=True,
    )


def encode_training_example(example: dict, tok) -> dict:
    """Prompt tokens (including prior assistant turns in context) get label -100; loss is computed only on target + EOS."""
    prompt_text = tok.apply_chat_template(
        build_chat_messages(example["context"]),
        tokenize=False,
        add_generation_prompt=True,
    )
    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    target_text = example["target"]["text"].strip()
    target_ids = tok(target_text + tok.eos_token, add_special_tokens=False)["input_ids"]

    max_target = MAX_SEQ_LEN - 32
    if len(target_ids) > max_target:
        target_ids = target_ids[:max_target] + [tok.eos_token_id]

    budget = MAX_SEQ_LEN - len(target_ids)
    if len(prompt_ids) > budget:
        # Preserve the start of the system prompt and the most recent context; trim the middle if needed.
        head = min(96, max(0, budget // 4))
        tail = max(0, budget - head)
        prompt_ids = prompt_ids[:head] + (prompt_ids[-tail:] if tail else [])

    labels = [-100] * len(prompt_ids) + target_ids
    assert len(prompt_ids) + len(target_ids) <= MAX_SEQ_LEN
    assert all(x == -100 for x in labels[:len(prompt_ids)])
    assert all(x != -100 for x in labels[len(prompt_ids):])
    return {
        "input_ids": prompt_ids + target_ids,
        "labels": labels,
        "target_tokens": len(target_ids),
    }


class CausalDataset(torch.utils.data.Dataset):
    def __init__(self, encoded_examples: list[dict]):
        self.items = encoded_examples

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


class CausalCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features: list[dict]) -> dict:
        max_len = max(len(feature["input_ids"]) for feature in features)
        input_ids, attention_mask, labels = [], [], []
        for feature in features:
            pad_len = max_len - len(feature["input_ids"])
            input_ids.append(feature["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(feature["input_ids"]) + [0] * pad_len)
            labels.append(feature["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def select_fewshot_examples(train_examples: list[dict], n: int = FEWSHOT_N) -> list[dict]:
    candidates = [
        ex for ex in train_examples
        if 4 <= len(ex["target"]["text"]) <= 120 and 2 <= len(ex["context"]) <= 6
    ]
    if len(candidates) < n:
        candidates = list(train_examples)
    rng = random.Random(SEED)
    return rng.sample(candidates, min(n, len(candidates)))


def select_test_cases(test_examples: list[dict], n: int = N_TEST_CASES) -> list[dict]:
    candidates = [
        ex for ex in test_examples
        if 3 <= len(ex["target"]["text"]) <= 180 and len(ex["context"]) >= 3
    ]
    if not candidates:
        candidates = list(test_examples)
    if len(candidates) <= n:
        return candidates
    indices = [round(i * (len(candidates) - 1) / (n - 1)) for i in range(n)]
    return [candidates[i] for i in indices]


fewshot_examples = select_fewshot_examples(examples_by_split["train"])
test_cases = select_test_cases(examples_by_split["test"])

probe = encode_training_example(examples_by_split["train"][0], tokenizer)
active = sum(1 for label in probe["labels"] if label != -100)
print("Few-shot examples:", len(fewshot_examples))
print("Test cases:", len(test_cases))
print("Probe: active loss tokens", active, "of", len(probe["labels"]), "(target + EOS only)")


In [ ]:
# ===== Shared Generation Helper =====
def clean_generated(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(?:ME|FRIEND|Assistant|Asystent)\s*:\s*", "", text, flags=re.I)
    return text


@torch.inference_mode()
def generate_reply(model, tok, context: list[dict], fewshot_examples: list[dict] | None = None) -> str:
    prompt = prompt_text_for(context, fewshot_examples)
    ids = tok(prompt, add_special_tokens=False)["input_ids"]
    if len(ids) > MAX_CONTEXT_TOKENS:
        head = 96
        ids = ids[:head] + ids[-(MAX_CONTEXT_TOKENS - head):]
    input_ids = torch.tensor([ids], device=model.device)
    attention_mask = torch.ones_like(input_ids)
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        **GEN_KWARGS,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    new_tokens = output[0, input_ids.shape[1]:]
    reply = tok.decode(new_tokens, skip_special_tokens=True)
    return anonymize_text(clean_generated(reply))


print("Generation helper ready; shared kwargs:", GEN_KWARGS)


## Variants 1 & 2: Base and Few-Shot


In [ ]:
# ===== Variants 1 & 2: Base Model and Few-Shot =====
base_replies = []
few_replies = []

model = load_model_with_optional_adapter(None)
t0 = time.time()
for case in test_cases:
    base_replies.append(generate_reply(model, tokenizer, case["context"]))
    few_replies.append(generate_reply(model, tokenizer, case["context"], fewshot_examples))
elapsed = time.time() - t0
print("Generated base + few-shot replies in", round(elapsed, 1), "s")
if torch.cuda.is_available():
    print("VRAM peak:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## QLoRA Fine-Tuning

Loss is computed exclusively on the target reply tokens. Prompt tokens — including prior `assistant` turns inside the conversation context — have label `-100`, so they do not contribute to the gradient. Trained adapters are saved locally under `artifacts/`.

Below: the training helper, a 2-step smoke test, an optional pilot run, and the full training cell.


In [ ]:
# ===== QLoRA Training Function =====
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


def train_lora(train_examples: list[dict], output_dir, max_steps=None, num_epochs=1,
               max_examples=None, seed=SEED, resume_from_checkpoint=False) -> dict:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    examples = list(train_examples)
    if max_examples is not None and max_examples < len(examples):
        examples = random.Random(seed).sample(examples, max_examples)

    tok = load_tokenizer()
    encoded = [encode_training_example(ex, tok) for ex in examples]
    dataset = CausalDataset(encoded)
    collator = CausalCollator(tok.pad_token_id)

    model = None
    trainer = None
    try:
        model = load_base_model()
        if torch.cuda.is_available():
            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
        )
        model = get_peft_model(model, lora_config)
        model.config.use_cache = False
        model.print_trainable_parameters()

        args = TrainingArguments(
            output_dir=str(output_dir),
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=num_epochs,
            max_steps=max_steps if max_steps is not None else -1,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_steps=5,
            logging_steps=5,
            logging_first_step=True,
            save_strategy="steps",
            save_steps=250,
            save_total_limit=2,
            eval_strategy="no",
            report_to="none",
            bf16=torch.cuda.is_available(),
            fp16=False,
            optim="adamw_8bit",
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            max_grad_norm=0.3,
            remove_unused_columns=False,
            label_names=["labels"],
            dataloader_num_workers=0,
            seed=seed,
            data_seed=seed,
        )
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=dataset,
            data_collator=collator,
        )
        resume_ckpt = None
        if resume_from_checkpoint == "auto":
            resume_ckpt = get_last_checkpoint(str(output_dir))
            if resume_ckpt is not None:
                print("Resuming training from checkpoint:", resume_ckpt)
        elif resume_from_checkpoint:
            resume_ckpt = resume_from_checkpoint

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
        elapsed = time.time() - t0
        model.save_pretrained(str(output_dir))
        tok.save_pretrained(str(output_dir))
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return {
            "output_dir": str(output_dir),
            "n_examples": len(examples),
            "max_steps": max_steps if max_steps is not None else "full epoch(s)",
            "loss": float(train_result.training_loss),
            "seconds": round(elapsed, 1),
            "trainable_params": int(trainable),
            "vram_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None,
        }
    finally:
        if trainer is not None:
            del trainer
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass


In [ ]:
# ===== Smoke Test (2 steps, 16 examples) =====
SMOKE_DIR = ARTIFACT_DIR / "lora_smoke"
smoke_result = None
try:
    smoke_result = train_lora(
        examples_by_split["train"],
        SMOKE_DIR,
        max_steps=2,
        num_epochs=1,
        max_examples=16,
    )
    print("Smoke test training OK.")
except Exception as exc:
    smoke_result = {"error": f"{type(exc).__name__}: {exc}"}
    print("SMOKE TEST FAILED:", smoke_result["error"])

display(pd.DataFrame([smoke_result]))
TRAINING_WORKS = "error" not in smoke_result

if TRAINING_WORKS:
    model = load_model_with_optional_adapter(SMOKE_DIR)
    smoke_reply = generate_reply(model, tokenizer, test_cases[0]["context"])
    print("Smoke: adapter reloaded; reply length:", len(smoke_reply), "chars.")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ===== Pilot Training (800 examples, 1 epoch) =====
RUN_PILOT_TRAINING = True
PILOT_TRAIN_EXAMPLES = 800
PILOT_EPOCHS = 1
PILOT_DIR = ARTIFACT_DIR / "lora_pilot"

if not TRAINING_WORKS:
    print("Skipped: smoke test did not pass.")
elif RUN_PILOT_TRAINING:
    pilot_result = train_lora(
        examples_by_split["train"],
        PILOT_DIR,
        max_steps=None,
        num_epochs=PILOT_EPOCHS,
        max_examples=PILOT_TRAIN_EXAMPLES,
    )
    display(pd.DataFrame([pilot_result]))
else:
    print("Pilot training skipped.")


In [ ]:
# ===== Full Training — toggle RUN_FULL_TRAINING = True to run =====
# On an 8 GB VRAM GPU (e.g., RTX 4060 Laptop), 3 epochs on ~10k examples take ~2-3 hours.
# Automatic checkpoint resuming is enabled via resume_from_checkpoint="auto".
RUN_FULL_TRAINING = False
FULL_DIR = ARTIFACT_DIR / "lora_full"

if RUN_FULL_TRAINING:
    full_result = train_lora(
        examples_by_split["train"],
        FULL_DIR,
        max_steps=None,
        num_epochs=3,
        max_examples=None,
        resume_from_checkpoint="auto",
    )
    display(pd.DataFrame([full_result]))
else:
    print("Full training skipped. Set RUN_FULL_TRAINING = True and run this cell.")


In [ ]:
# ===== Variant 3: LoRA Evaluation & Side-by-Side Comparison =====
ADAPTER_CANDIDATES = [
    (ARTIFACT_DIR / "lora_full", "LoRA (full)"),
    (ARTIFACT_DIR / "lora_pilot", "LoRA (pilot)"),
    (ARTIFACT_DIR / "lora_smoke", "LoRA (smoke)"),
]
ADAPTER_FOR_EVAL = None
ADAPTER_LABEL = "LoRA (no adapter)"
for adapter_path, adapter_label in ADAPTER_CANDIDATES:
    if (adapter_path / "adapter_config.json").exists():
        ADAPTER_FOR_EVAL = adapter_path
        ADAPTER_LABEL = adapter_label
        break

print("Adapter selected for evaluation:", ADAPTER_LABEL, "|", ADAPTER_FOR_EVAL)

lora_replies = []
if ADAPTER_FOR_EVAL is not None:
    model = load_model_with_optional_adapter(ADAPTER_FOR_EVAL)
    for case in test_cases:
        lora_replies.append(generate_reply(model, tokenizer, case["context"]))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    lora_replies = ["(no adapter found — run training first)"] * len(test_cases)

rows_html = []
for i, case in enumerate(test_cases, start=1):
    rows_html.append(
        "<tr>"
        f"<td><b>{i}</b><br><small>{html.escape(case['date'])}</small></td>"
        f"<td>{_pre(format_context(case['context']))}</td>"
        f"<td>{_pre(case['target']['text'])}</td>"
        f"<td>{_pre(base_replies[i-1])}</td>"
        f"<td>{_pre(few_replies[i-1])}</td>"
        f"<td>{_pre(lora_replies[i-1])}</td>"
        "</tr>"
    )

display(HTML(
    f"<h4>Side-by-Side Replies on {len(test_cases)} Held-Out Test Cases</h4>"
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
    "<tr><th style='text-align:left;padding:6px'>#</th>"
    "<th style='text-align:left;padding:6px'>context (anonymized)</th>"
    "<th style='text-align:left;padding:6px'>ground-truth reply</th>"
    "<th style='text-align:left;padding:6px'>base</th>"
    "<th style='text-align:left;padding:6px'>few-shot</th>"
    f"<th style='text-align:left;padding:6px'>{html.escape(ADAPTER_LABEL)}</th></tr>"
    + "".join(rows_html) +
    "</table>"
    "<style>td,th{border:1px solid #ccc;vertical-align:top;padding:6px;white-space:pre-wrap}</style>"
))


### Manual Evaluation Rubric

Rate each generated reply from **1 to 5** on two dimensions:

- **Sounds like me**: 1 = completely generic/robotic, 3 = partially captures style, 5 = closely matches personal phrasing and rhythm.
- **Fits conversation**: 1 = ignores context or goes off-topic, 3 = plausible but generic, 5 = directly fits the exchange.

Record scores in `MANUAL_SCORES` below (e.g., `MANUAL_SCORES[(1, "LoRA")] = {"sounds_like_me": 4, "fits_conversation": 4}`). Length and line-count tables are diagnostic only — they do not prove style fidelity.


In [ ]:
# ===== Manual Scores & Length/Line-Count Diagnostics =====
MANUAL_SCORES = {
    # (case_index, "base" / "few-shot" / "LoRA"): {"sounds_like_me": 4, "fits_conversation": 4},
}

variants = ["base", "few-shot", "LoRA"]
replies_by_variant = {
    "base": base_replies,
    "few-shot": few_replies,
    "LoRA": lora_replies,
}

manual_rows = []
for i in range(len(test_cases)):
    for variant in variants:
        score = MANUAL_SCORES.get((i + 1, variant), {})
        manual_rows.append({
            "case": i + 1,
            "variant": variant,
            "sounds_like_me (1-5)": score.get("sounds_like_me", pd.NA),
            "fits_conversation (1-5)": score.get("fits_conversation", pd.NA),
        })
display(pd.DataFrame(manual_rows))


def length_stats(text: str) -> tuple[int, int]:
    if not text:
        return 0, 0
    return len(text), text.count("\n") + 1


length_rows = []
for i, case in enumerate(test_cases, start=1):
    true_text = case["target"]["text"]
    row = {"case": i, "true_chars": len(true_text), "true_lines": length_stats(true_text)[1]}
    for variant in variants:
        n_chars, n_lines = length_stats(replies_by_variant[variant][i - 1])
        row[f"{variant}_chars"] = n_chars
        row[f"{variant}_lines"] = n_lines
    length_rows.append(row)

display(pd.DataFrame(length_rows))

summary_rows = []
for variant in variants:
    texts = replies_by_variant[variant]
    summary_rows.append({
        "variant": variant,
        "avg_chars": round(sum(len(t) for t in texts) / max(1, len(texts)), 1),
        "avg_lines": round(sum(length_stats(t)[1] for t in texts) / max(1, len(texts)), 2),
        "length_ratio_vs_true": round(
            sum(len(t) for t in texts) / max(1, sum(len(c["target"]["text"]) for c in test_cases)), 2
        ),
    })
display(pd.DataFrame(summary_rows))

print("Note: length and line count are diagnostics only; they do not prove stylistic accuracy.")


## Interactive Playground

Enter any conversation snippet using `FRIEND:` and `ME:` prefixes (subsequent lines without a prefix belong to the active speaker). This cell caches the loaded adapter in memory so repeated runs with edited prompts are instantaneous.


In [ ]:
# ===== Interactive Playground with Trained LoRA Adapter =====
import re
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

PLAYGROUND_ADAPTER_CANDIDATES = [
    Path("artifacts/lora_full"),
    Path("artifacts/lora_pilot"),
    Path("artifacts/lora_smoke"),
]
ADAPTER_PATH = next((p for p in PLAYGROUND_ADAPTER_CANDIDATES if (p / "adapter_config.json").exists()), None)
if ADAPTER_PATH is None:
    raise FileNotFoundError("No trained LoRA adapter found in artifacts/. Run training first.")

BASE_MODEL_ID = globals().get("MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")
PLAYGROUND_SYSTEM_PROMPT = globals().get(
    "SYSTEM_PROMPT",
    "You are a person referred to as ME writing private chat messages to FRIEND. "
    "Mimic the author's natural messaging style: message length, slang, emojis, typos, "
    "punctuation, and line breaks. Reply only with the message content — no commentary, "
    "no metadata, and no invented personal details.",
)

if "playground_model" not in globals() or globals().get("playground_adapter") != str(ADAPTER_PATH):
    tok_src = str(ADAPTER_PATH) if (ADAPTER_PATH / "tokenizer_config.json").exists() else BASE_MODEL_ID
    playground_tokenizer = AutoTokenizer.from_pretrained(tok_src)
    if playground_tokenizer.pad_token is None:
        playground_tokenizer.pad_token = playground_tokenizer.eos_token
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_cfg if torch.cuda.is_available() else None,
        device_map={"": 0} if torch.cuda.is_available() else None,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    playground_model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
    playground_model.eval()
    playground_adapter = str(ADAPTER_PATH)
    print("Loaded adapter from:", ADAPTER_PATH)

# --- Edit the conversation snippet below ---
CONVERSATION = """
FRIEND: hey, are we still working on that project tonight or tomorrow?
ME: tonight works for me but only after 7pm
FRIEND: cool, 7:30 on discord or in person?
"""

# Set DO_SAMPLE = True for more varied, stochastic replies.
DO_SAMPLE = False
TEMPERATURE = 0.7
TOP_P = 0.9


def parse_conversation(raw_text: str) -> list[dict]:
    messages = [{"role": "system", "content": PLAYGROUND_SYSTEM_PROMPT}]
    current_role = None
    buffer = []
    for line in raw_text.strip().splitlines():
        m = re.match(r"^(FRIEND|ME)\s*:\s*(.*)$", line.strip(), flags=re.I)
        if m:
            if current_role is not None and buffer:
                messages.append({"role": current_role, "content": "\n".join(buffer).strip()})
            speaker = m.group(1).upper()
            current_role = "user" if speaker == "FRIEND" else "assistant"
            buffer = [m.group(2)]
        elif current_role is not None:
            buffer.append(line.rstrip())
    if current_role is not None and buffer:
        messages.append({"role": current_role, "content": "\n".join(buffer).strip()})
    return messages


@torch.inference_mode()
def predict_reply(raw_text: str, do_sample: bool = DO_SAMPLE) -> str:
    msgs = parse_conversation(raw_text)
    prompt = playground_tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = playground_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(playground_model.device)
    gen_cfg = {
        "max_new_tokens": 96,
        "do_sample": do_sample,
        "repetition_penalty": 1.1,
        "no_repeat_ngram_size": 4,
        "pad_token_id": playground_tokenizer.pad_token_id,
        "eos_token_id": playground_tokenizer.eos_token_id,
    }
    if do_sample:
        gen_cfg["temperature"] = TEMPERATURE
        gen_cfg["top_p"] = TOP_P
    out = playground_model.generate(**inputs, **gen_cfg)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    reply = playground_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return re.sub(r"^(?:ME|FRIEND|Assistant)\s*:\s*", "", reply, flags=re.I)


reply = predict_reply(CONVERSATION)
print("=== Model Reply (ME) ===")
print(reply)
